# LeetCode 531 — Lonely Pixel I bằng Python

Link đề: https://leetcode.com/problems/lonely-pixel-i/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Matrix

Skeleton LeetCode:

```python
class Solution:
    def findLonelyPixel(self, picture: List[List[str]]) -> int:
        ...
```

Ghi chú: đây là bài **Premium**. Đề trong notebook được viết theo hiểu biết về bài (đối chiếu với bản mô tả công khai), nên nếu bạn có tài khoản Premium hãy đối chiếu lại đề gốc, đặc biệt phần ví dụ và ràng buộc. Các đáp án trong notebook đều được kiểm chứng bằng code (vét cạn + so sánh nhiều cách).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong một bức ảnh `m x n` gồm các điểm ảnh đen `'B'` và trắng `'W'`, đếm số điểm đen **cô đơn** — điểm `'B'` mà trên **cùng hàng** và **cùng cột** của nó **không có điểm `'B'` nào khác**.
2. Nắm các công cụ Python cho bài ma trận: `list` lồng `list`, `enumerate`, `str.count`, `zip(*picture)` để lấy các cột, `collections.Counter`, `sum` với generator.
3. Đi từ **vét cạn (quét lại hàng và cột cho từng điểm đen)** → **đếm số `'B'` theo hàng và theo cột rồi kiểm tra** → **đếm theo cột và chỉ kiểm tra cột có đúng một `'B'`** → **cách "Pythonic" với `count` và `zip`** → **chỉ lưu tọa độ các điểm đen với `Counter`**.
4. Hiểu mấu chốt: một điểm là cô đơn khi **hai con số** (số `'B'` trên hàng, số `'B'` trên cột) **đều bằng 1**; hai con số này chỉ cần tính **một lần cho cả hàng/cột** thay vì tính lại cho từng điểm.
5. Tránh các lỗi kinh điển: chỉ kiểm tra hàng mà quên cột, dùng `or` thay vì `and`, nhầm kích thước (`m` và `n`) khi tạo mảng đếm, chỉ nhìn các ô hàng xóm kề bên, và quên rằng chính điểm đang xét cũng được đếm.

## 2. Đề bài & ví dụ

Cho một ma trận `picture` kích thước `m x n` chỉ gồm hai ký tự: `'B'` (điểm ảnh **đen**, black) và `'W'` (điểm ảnh **trắng**, white). Hãy trả về **số điểm đen cô đơn** (lonely black pixels).

**Điểm đen cô đơn** là một ký tự `'B'` ở vị trí `(i, j)` sao cho **trên hàng `i` và trên cột `j` không có điểm `'B'` nào khác** ngoài chính nó. (Điểm ở hàng `i` và cột `j` khác không quan trọng, chỉ cần không có `'B'` khác cùng hàng **hoặc** cùng cột.)

**Ví dụ 1**

```
W W B
W B W
B W W
```

- Input: `picture = [["W","W","B"],["W","B","W"],["B","W","W"]]`
- Output: `3`
- Giải thích: ba điểm `'B'` nằm trên ba hàng khác nhau và ba cột khác nhau, nên cả ba đều cô đơn.

**Ví dụ 2**

```
B B B
B B W
B B B
```

- Input: `picture = [["B","B","B"],["B","B","W"],["B","B","B"]]`
- Output: `0`
- Giải thích: mỗi `'B'` đều có `'B'` khác cùng hàng hoặc cùng cột, không có điểm nào cô đơn.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `picture = [["B","W","W"],["W","W","W"],["W","W","W"]]` (chỉ một `'B'`)
- Output: `1`

**Ràng buộc:**

- `m == picture.length`, `n == picture[i].length`
- `1 <= m, n <= 500`
- `picture[i][j]` là `'W'` hoặc `'B'`.

Chú ý: ma trận không bao giờ rỗng (`m, n >= 1`), nhưng có thể **không có `'B'` nào** (đáp án 0) hoặc **toàn `'B'`** (đáp án 0, trừ trường hợp `1 x 1`).

In [1]:
from typing import List
from collections import Counter, defaultdict
import random
import time


def show(picture):
    """In ma trận cho dễ nhìn."""
    for row in picture:
        print(" ".join(row))


examples = [
    ([["W", "W", "B"], ["W", "B", "W"], ["B", "W", "W"]], 3),
    ([["B", "B", "B"], ["B", "B", "W"], ["B", "B", "B"]], 0),
    ([["B", "W", "W"], ["W", "W", "W"], ["W", "W", "W"]], 1),
]
for pic, expected in examples:
    show(pic)
    print("-> kỳ vọng", expected)
    print()

W W B
W B W
B W W
-> kỳ vọng 3

B B B
B B W
B B B
-> kỳ vọng 0

B W W
W W W
W W W
-> kỳ vọng 1



## 3. Kiến thức nền

### 3.1. Ma trận là "list của các list"

Trong Python, ma trận `m x n` thường là một `list` có `m` phần tử, mỗi phần tử là một `list` có `n` ký tự. Phần tử ở **hàng `i`, cột `j`** là `picture[i][j]`. Số hàng là `len(picture)`, số cột là `len(picture[0])` (vì đề đảm bảo mọi hàng cùng độ dài và `m >= 1`).

In [2]:
picture = [["W", "W", "B"],
           ["W", "B", "W"],
           ["B", "W", "W"]]

m = len(picture)         # số hàng
n = len(picture[0])      # số cột
print("m =", m, "| n =", n)
print("picture[0] =", picture[0], "(cả hàng 0)")
print("picture[1][1] =", picture[1][1], "(hàng 1, cột 1)")
print("picture[2][0] =", picture[2][0], "(hàng 2, cột 0)")

m = 3 | n = 3
picture[0] = ['W', 'W', 'B'] (cả hàng 0)
picture[1][1] = B (hàng 1, cột 1)
picture[2][0] = B (hàng 2, cột 0)


### 3.2. Duyệt hai vòng `for` với `enumerate`

Để đi qua mọi ô, ta lồng hai vòng lặp. `enumerate(row)` cho ta **cả chỉ số và giá trị** trong một lần, tiện hơn `range(len(...))`.

In [3]:
for i, row in enumerate(picture):
    for j, x in enumerate(row):
        if x == "B":
            print(f"có điểm đen ở hàng {i}, cột {j}")

có điểm đen ở hàng 0, cột 2
có điểm đen ở hàng 1, cột 1
có điểm đen ở hàng 2, cột 0


### 3.3. `str.count`, `list.count` và `list.index`

- `row.count("B")` đếm số lần `"B"` xuất hiện trong `row` (một `list` ký tự, hoặc một `str`). Tốn `O(n)` nhưng chạy bằng mã C nên rất nhanh.
- `row.index("B")` trả về vị trí **đầu tiên** của `"B"` (báo lỗi `ValueError` nếu không có). Khi ta biết hàng có **đúng một** `'B'`, `index` cho ngay cột của nó.

In [4]:
row = ["W", "B", "W", "B"]
print("số B:", row.count("B"))
print("vị trí B đầu tiên:", row.index("B"))

only_one = ["W", "W", "B"]
print("hàng có đúng 1 B -> cột của nó:", only_one.index("B"))

try:
    ["W", "W"].index("B")
except ValueError as e:
    print("không có B thì index báo lỗi:", e)

số B: 2
vị trí B đầu tiên: 1
hàng có đúng 1 B -> cột của nó: 2
không có B thì index báo lỗi: 'B' is not in list


### 3.4. `zip(*picture)` để lấy các **cột**

Với list các hàng, `zip(*picture)` "xoay" ma trận: mỗi phần tử trả ra là **một cột** (dạng `tuple`). Dấu `*` trải list `picture` thành các đối số riêng lẻ `picture[0], picture[1], ...` rồi `zip` ghép phần tử cùng vị trí.

In [5]:
for j, col in enumerate(zip(*picture)):
    print(f"cột {j}:", col, "| số B =", col.count("B"))

cột 0: ('W', 'W', 'B') | số B = 1
cột 1: ('W', 'B', 'W') | số B = 1
cột 2: ('B', 'W', 'W') | số B = 1


### 3.5. `Counter` và `defaultdict`

`collections.Counter` đếm số lần xuất hiện của từng giá trị, và trả `0` cho giá trị chưa gặp (không báo lỗi). Ta dùng nó để đếm "hàng `i` có bao nhiêu điểm đen" khi chỉ giữ danh sách tọa độ.

In [6]:
blacks = [(0, 2), (1, 1), (2, 0), (2, 2)]
row_count = Counter(i for i, _ in blacks)
col_count = Counter(j for _, j in blacks)
print("rows:", row_count)
print("cols:", col_count)
print("hàng 5 (chưa gặp) ->", row_count[5])

rows: Counter({2: 2, 0: 1, 1: 1})
cols: Counter({2: 2, 1: 1, 0: 1})
hàng 5 (chưa gặp) -> 0


### 3.6. `sum` với generator, `all` và `any`

- `sum(1 for ... if điều_kiện)` là cách gọn để **đếm** số phần tử thỏa điều kiện.
- `any(...)` trả `True` nếu **có ít nhất một** phần tử thỏa; `all(...)` trả `True` nếu **mọi** phần tử thỏa.

In [7]:
nums = [3, 8, 1, 6]
print("đếm số lớn hơn 2:", sum(1 for x in nums if x > 2))
print("any x > 7:", any(x > 7 for x in nums))
print("all x > 7:", all(x > 7 for x in nums))
print("any trên list rỗng:", any([]), "| all trên list rỗng:", all([]))

đếm số lớn hơn 2: 3
any x > 7: True
all x > 7: False
any trên list rỗng: False | all trên list rỗng: True


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Hãy tự làm bằng tay với ma trận `4 x 4` sau (`.` là `W` cho dễ nhìn):

```
      c0 c1 c2 c3
r0:    B  .  .  .
r1:    .  .  B  .
r2:    .  B  .  B
r3:    .  .  .  .
```

Có 4 điểm đen: `(0,0)`, `(1,2)`, `(2,1)`, `(2,3)`. Xét từng điểm:

- `(0,0)`: hàng 0 chỉ có nó; cột 0 chỉ có nó → **cô đơn**.
- `(1,2)`: hàng 1 chỉ có nó; cột 2 chỉ có nó → **cô đơn**.
- `(2,1)`: hàng 2 còn `(2,3)` → **không** cô đơn.
- `(2,3)`: hàng 2 còn `(2,1)` → **không** cô đơn.

Đáp án là `2`.

Nhận xét quan trọng: để trả lời "điểm này có cô đơn không?" ta chỉ cần biết **hàng của nó có bao nhiêu `B`** và **cột của nó có bao nhiêu `B`**. Nếu cả hai đều bằng **1** (chính nó) thì cô đơn. Lập bảng đếm:

| | c0 | c1 | c2 | c3 | số B của hàng |
|---|---|---|---|---|---|
| r0 | B | . | . | . | 1 |
| r1 | . | . | B | . | 1 |
| r2 | . | B | . | B | 2 |
| r3 | . | . | . | . | 0 |
| **số B của cột** | 1 | 1 | 1 | 1 | |

In [8]:
grid4 = [list("B..."), list("..B."), list(".B.B"), list("....")]
# đổi '.' thành 'W' cho đúng đề
grid4 = [["W" if x == "." else x for x in row] for row in grid4]

row_counts = [row.count("B") for row in grid4]
col_counts = [col.count("B") for col in zip(*grid4)]
print("số B mỗi hàng:", row_counts)
print("số B mỗi cột :", col_counts)

lonely = [(i, j) for i, row in enumerate(grid4) for j, x in enumerate(row)
          if x == "B" and row_counts[i] == 1 and col_counts[j] == 1]
print("điểm cô đơn:", lonely, "-> đáp án", len(lonely))

số B mỗi hàng: [1, 1, 2, 0]
số B mỗi cột : [1, 1, 1, 1]
điểm cô đơn: [(0, 0), (1, 2)] -> đáp án 2


## 5. Cách 1: Brute Force (vét cạn)

### Ý tưởng

Làm đúng theo định nghĩa: với **mỗi** ô `'B'` tại `(i, j)`:

1. Quét cả hàng `i`: nếu thấy `'B'` ở cột `c != j` thì điểm này không cô đơn.
2. Quét cả cột `j`: nếu thấy `'B'` ở hàng `r != i` thì điểm này không cô đơn.
3. Nếu hai lượt quét đều không thấy gì → cô đơn, tăng đáp án.

In [9]:
def lonely_brute(picture):
    m, n = len(picture), len(picture[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if picture[i][j] != "B":
                continue
            alone = True
            for c in range(n):                       # quét hàng i
                if c != j and picture[i][c] == "B":
                    alone = False
                    break
            if alone:
                for r in range(m):                   # quét cột j
                    if r != i and picture[r][j] == "B":
                        alone = False
                        break
            if alone:
                ans += 1
    return ans


for pic, expected in examples:
    got = lonely_brute(pic)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )
1 (kỳ vọng 1 )


### Độ phức tạp

- Mỗi điểm đen tốn tối đa `O(m + n)` để quét hàng và cột; có tối đa `m * n` điểm đen ⇒ cận trên thô là `O(m * n * (m + n))`. Với `m = n = 500` đó là cỡ `250.000 * 1.000 = 2,5 * 10^8` bước trong trường hợp xấu — quá chậm với Python.
- Bộ nhớ phụ: `O(1)`.

### Nút thắt

Cùng một hàng bị quét **lại nhiều lần**: hàng có `k` điểm đen thì bị quét `k` lần, mỗi lần chỉ để biết "hàng này có hơn một `B` hay không" — một câu hỏi mà **câu trả lời không phụ thuộc vào điểm ta đang xét**. Cột cũng vậy. Hãy đếm xem tốn bao nhiêu phép so sánh với một ma trận cụ thể:

In [10]:
def lonely_brute_counting(picture):
    """Giống lonely_brute nhưng đếm số lần đọc ô picture[..][..]."""
    m, n = len(picture), len(picture[0])
    reads = 0
    ans = 0
    for i in range(m):
        for j in range(n):
            if picture[i][j] != "B":
                continue
            alone = True
            for c in range(n):
                if c != j:
                    reads += 1
                    if picture[i][c] == "B":
                        alone = False
                        break
            if alone:
                for r in range(m):
                    if r != i:
                        reads += 1
                        if picture[r][j] == "B":
                            alone = False
                            break
            ans += alone
    return ans, reads


# ma trận chéo: mọi điểm đều cô đơn -> mỗi điểm quét đủ cả hàng lẫn cột
for size in (10, 50, 100, 200):
    diag = [["B" if i == j else "W" for j in range(size)] for i in range(size)]
    ans, reads = lonely_brute_counting(diag)
    print(f"{size:>3} x {size:<3} đường chéo: đáp án {ans:>3}, số lần đọc ô = {reads:>8}")

 10 x 10  đường chéo: đáp án  10, số lần đọc ô =      180
 50 x 50  đường chéo: đáp án  50, số lần đọc ô =     4900
100 x 100 đường chéo: đáp án 100, số lần đọc ô =    19800
200 x 200 đường chéo: đáp án 200, số lần đọc ô =    79600


Số lần đọc tăng cỡ **bình phương** của kích thước ngay cả khi chỉ có `size` điểm đen. Nếu ma trận có nhiều điểm đen hơn thì còn nặng hơn nữa. Ta cần cách trả lời "hàng/cột này có mấy `B`?" **một lần cho cả hàng/cột**.

## 6. Ý tưởng thuật toán / bước đột phá

Câu hỏi "hàng `i` có bao nhiêu điểm đen?" chỉ phụ thuộc vào **`i`**, không phụ thuộc vào điểm cụ thể ta đang xét. Vậy hãy **tính trước** một lần:

- `rows[i]` = số `'B'` trên hàng `i`
- `cols[j]` = số `'B'` trên cột `j`

Sau đó điểm `(i, j)` là cô đơn **khi và chỉ khi** `picture[i][j] == 'B'` và `rows[i] == 1` và `cols[j] == 1`. (Vì `rows[i] == 1` mà chính nó là một `'B'` ở hàng `i` nghĩa là nó là `'B'` duy nhất.)

Đây là kiểu "**tính trước thống kê rồi tra cứu**" (precompute + lookup): đổi `O(m + n)` mỗi lần kiểm tra thành `O(1)`. Tổng thời gian còn `O(m * n)` — vừa đủ để *đọc* mỗi ô một hoặc hai lần — và đó là tối ưu về bậc vì ta phải đọc cả ma trận.

In [11]:
# minh họa: chỉ cần hai mảng đếm là trả lời được mọi điểm trong O(1)
pic = examples[0][0]
rows = [r.count("B") for r in pic]
cols = [c.count("B") for c in zip(*pic)]
print("rows =", rows, "| cols =", cols)
for i in range(3):
    for j in range(3):
        if pic[i][j] == "B":
            print(f"({i},{j}): rows[{i}]={rows[i]}, cols[{j}]={cols[j]} -> cô đơn?", rows[i] == 1 and cols[j] == 1)

rows = [1, 1, 1] | cols = [1, 1, 1]
(0,2): rows[0]=1, cols[2]=1 -> cô đơn? True
(1,1): rows[1]=1, cols[1]=1 -> cô đơn? True
(2,0): rows[2]=1, cols[0]=1 -> cô đơn? True


## 7. Các cách tiếp theo

Ngoài vét cạn (cách 1) ta có thêm **4 cách** khác nhau về ý tưởng:

- **Cách 2:** hai mảng đếm `rows`, `cols` (đếm trong lượt 1, kiểm tra trong lượt 2).
- **Cách 3:** đếm theo cột, nhớ "hàng của `B` trong cột", rồi chỉ kiểm tra **các cột** có đúng một `B`.
- **Cách 4:** "Pythonic" — dùng `str.count`, `zip(*picture)` và `index`.
- **Cách 5:** chỉ lưu **tọa độ các điểm đen** và dùng `Counter`.

### 7.1. Cách 2 — Hai mảng đếm (đếm rồi kiểm tra)

**Giải thích từng đoạn:**

1. `rows = [0] * m`, `cols = [0] * n`: hai mảng đếm. Chú ý **`rows` có `m` phần tử (số hàng)**, **`cols` có `n` phần tử (số cột)**.
2. Lượt 1: gặp `'B'` ở `(i, j)` thì `rows[i] += 1`, `cols[j] += 1`.
3. Lượt 2: gặp `'B'` ở `(i, j)` mà `rows[i] == 1 and cols[j] == 1` thì tăng đáp án.

**Code demo (có in để quan sát):**

In [12]:
def lonely_count_demo(picture):
    m, n = len(picture), len(picture[0])
    rows = [0] * m
    cols = [0] * n
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                rows[i] += 1
                cols[j] += 1
    print("rows =", rows, "| cols =", cols)
    ans = 0
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B" and rows[i] == 1 and cols[j] == 1:
                print(f"  ({i},{j}) cô đơn")
                ans += 1
    return ans

print("đáp án:", lonely_count_demo(examples[0][0]))
print("đáp án:", lonely_count_demo(grid4))

rows = [1, 1, 1] | cols = [1, 1, 1]
  (0,2) cô đơn
  (1,1) cô đơn
  (2,0) cô đơn
đáp án: 3
rows = [1, 1, 2, 0] | cols = [1, 1, 1, 1]
  (0,0) cô đơn
  (1,2) cô đơn
đáp án: 2


**Code nộp LeetCode (cách 2):**

In [13]:
class Solution2:
    def findLonelyPixel(self, picture: List[List[str]]) -> int:
        rows = [0] * len(picture)
        cols = [0] * len(picture[0])
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B":
                    rows[i] += 1
                    cols[j] += 1
        ans = 0
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B" and rows[i] == 1 and cols[j] == 1:
                    ans += 1
        return ans


for pic, expected in examples:
    assert Solution2().findLonelyPixel(pic) == expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp:** Time `O(m * n)` (hai lượt duyệt toàn ma trận). Space `O(m + n)` cho hai mảng đếm.

### 7.2. Cách 3 — Đếm theo cột, chỉ kiểm tra các cột có đúng một `B`

**Ý tưởng khác:** mỗi điểm cô đơn nằm trong một cột có **đúng một** `'B'`. Vậy thay vì lượt 2 duyệt lại `m * n` ô, ta chỉ cần với **mỗi cột** nhớ thêm "hàng của `B` gần nhất" `last_row[j]`. Sau lượt duyệt duy nhất, ta xét `n` cột: cột `j` có `cols[j] == 1` thì `B` duy nhất của nó nằm ở hàng `last_row[j]`; ta chỉ cần kiểm tra `rows[last_row[j]] == 1`.

**Giải thích từng đoạn:**

1. Một lượt duyệt: với mỗi `'B'` ở `(i, j)`: `rows[i] += 1`, `cols[j] += 1`, `last_row[j] = i`.
2. Lượt thứ hai chỉ chạy trên `n` cột (không phải `m * n` ô).
3. Với cột `j` có `cols[j] == 1`: `last_row[j]` chính là hàng của `B` duy nhất, và nó cô đơn khi hàng đó cũng chỉ có một `B`.

**Code demo:**

In [14]:
def lonely_by_column_demo(picture):
    m, n = len(picture), len(picture[0])
    rows = [0] * m
    cols = [0] * n
    last_row = [-1] * n            # hàng của B được thấy gần nhất trong cột j
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                rows[i] += 1
                cols[j] += 1
                last_row[j] = i
    print("rows =", rows, "| cols =", cols, "| last_row =", last_row)
    ans = 0
    for j in range(n):
        if cols[j] == 1:
            i = last_row[j]
            ok = rows[i] == 1
            print(f"  cột {j} có đúng 1 B ở hàng {i}; rows[{i}] = {rows[i]} -> {'cô đơn' if ok else 'không'}")
            ans += ok
    return ans

print("đáp án:", lonely_by_column_demo(grid4))

rows = [1, 1, 2, 0] | cols = [1, 1, 1, 1] | last_row = [0, 2, 1, 2]
  cột 0 có đúng 1 B ở hàng 0; rows[0] = 1 -> cô đơn
  cột 1 có đúng 1 B ở hàng 2; rows[2] = 2 -> không
  cột 2 có đúng 1 B ở hàng 1; rows[1] = 1 -> cô đơn
  cột 3 có đúng 1 B ở hàng 2; rows[2] = 2 -> không
đáp án: 2


**Code nộp LeetCode (cách 3):**

In [15]:
class Solution3:
    def findLonelyPixel(self, picture: List[List[str]]) -> int:
        m, n = len(picture), len(picture[0])
        rows = [0] * m
        cols = [0] * n
        last_row = [0] * n
        for i in range(m):
            for j in range(n):
                if picture[i][j] == "B":
                    rows[i] += 1
                    cols[j] += 1
                    last_row[j] = i
        ans = 0
        for j in range(n):
            if cols[j] == 1 and rows[last_row[j]] == 1:
                ans += 1
        return ans


for pic, expected in examples:
    assert Solution3().findLonelyPixel(pic) == expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp:** Time `O(m * n)` cho lượt duyệt đọc ma trận + `O(n)` cho lượt kiểm tra cột (nhỏ hơn hẳn). Space `O(m + n)` (thêm `last_row` cỡ `n`). Cách này cùng bậc với cách 2 nhưng chỉ đọc ma trận **một lần**, nên hữu ích khi ma trận đến dưới dạng luồng (đọc một lần).

Vì sao `last_row[j]` đúng? Khi `cols[j] == 1`, chỉ có **một** lần gán `last_row[j] = i`, nên giá trị đó chính là hàng của `B` duy nhất. Khi `cols[j] != 1` ta không dùng đến `last_row[j]` nên giá trị của nó không quan trọng.

### 7.3. Cách 4 — "Pythonic": `count`, `zip(*picture)` và `index`

**Ý tưởng:** tận dụng thư viện chuẩn để viết ngắn. Điểm cô đơn nằm trong một **hàng có đúng một `B`**; với hàng như vậy, `row.index("B")` cho ngay cột của nó, chỉ cần tra `cols[cột] == 1`. Vậy ta không cần duyệt từng ô bằng hai vòng `for`.

**Giải thích từng đoạn:**

1. `rows = [row.count("B") for row in picture]`: số `B` mỗi hàng (mỗi `count` chạy nhanh bằng mã C).
2. `cols = [col.count("B") for col in zip(*picture)]`: số `B` mỗi cột nhờ `zip`.
3. Với mỗi hàng `i` có `rows[i] == 1`: cột của `B` là `picture[i].index("B")`; nếu `cols[cột] == 1` thì tính một điểm.

**Code demo:**

In [16]:
def lonely_pythonic_demo(picture):
    rows = [row.count("B") for row in picture]
    cols = [col.count("B") for col in zip(*picture)]
    print("rows =", rows, "| cols =", cols)
    ans = 0
    for i, row in enumerate(picture):
        if rows[i] == 1:
            j = row.index("B")
            print(f"  hàng {i} có đúng 1 B ở cột {j}; cols[{j}] = {cols[j]}")
            if cols[j] == 1:
                ans += 1
    return ans

print("đáp án:", lonely_pythonic_demo(grid4))

rows = [1, 1, 2, 0] | cols = [1, 1, 1, 1]
  hàng 0 có đúng 1 B ở cột 0; cols[0] = 1
  hàng 1 có đúng 1 B ở cột 2; cols[2] = 1
đáp án: 2


**Code nộp LeetCode (cách 4):**

In [17]:
class Solution4:
    def findLonelyPixel(self, picture: List[List[str]]) -> int:
        cols = [col.count("B") for col in zip(*picture)]
        ans = 0
        for row in picture:
            if row.count("B") == 1 and cols[row.index("B")] == 1:
                ans += 1
        return ans


for pic, expected in examples:
    assert Solution4().findLonelyPixel(pic) == expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp:** Time `O(m * n)` (mỗi hàng bị `count` một lần, mỗi cột bị `count` một lần, `index` thêm tối đa một lần mỗi hàng). Space `O(m + n)`: `zip` là "lười", mỗi cột là một `tuple` `m` phần tử được tạo lần lượt khi ta duyệt (đỉnh điểm `O(m)`), còn list `cols` chỉ giữ `n` số đếm; phép `*picture` trải ra `m` đối số (cũng `O(m)`). Ưu điểm: ngắn và nhanh trong thực tế vì `count`/`index` chạy bằng C.

### 7.4. Cách 5 — Chỉ lưu tọa độ các điểm đen + `Counter`

**Ý tưởng:** đổi góc nhìn — thay vì "ma trận", coi bức ảnh là **tập các tọa độ `(i, j)` của điểm đen**. Một điểm `(i, j)` cô đơn khi **trong tập đó, chỉ có một điểm có hoành độ `i` và chỉ có một điểm có tung độ `j`**. Dùng hai `Counter` để đếm.

Cách này đặc biệt hợp khi ma trận **thưa** (ít điểm đen): sau khi đã có danh sách tọa độ, mọi việc còn lại chỉ tốn `O(số điểm đen)`, không phụ thuộc `m * n`.

**Giải thích từng đoạn:**

1. `blacks = [(i, j) for ... if picture[i][j] == "B"]`: thu thập tọa độ (đây là lần duy nhất đọc toàn ma trận).
2. `row_cnt = Counter(i for i, _ in blacks)`, `col_cnt = Counter(j for _, j in blacks)`.
3. `ans = sum(1 for i, j in blacks if row_cnt[i] == 1 and col_cnt[j] == 1)`.

**Code demo:**

In [18]:
def lonely_sparse_demo(picture):
    blacks = [(i, j) for i, row in enumerate(picture) for j, x in enumerate(row) if x == "B"]
    row_cnt = Counter(i for i, _ in blacks)
    col_cnt = Counter(j for _, j in blacks)
    print("blacks =", blacks)
    print("row_cnt =", dict(row_cnt), "| col_cnt =", dict(col_cnt))
    return sum(1 for i, j in blacks if row_cnt[i] == 1 and col_cnt[j] == 1)

print("đáp án:", lonely_sparse_demo(grid4))

blacks = [(0, 0), (1, 2), (2, 1), (2, 3)]
row_cnt = {0: 1, 1: 1, 2: 2} | col_cnt = {0: 1, 2: 1, 1: 1, 3: 1}
đáp án: 2


**Code nộp LeetCode (cách 5):**

In [19]:
class Solution5:
    def findLonelyPixel(self, picture: List[List[str]]) -> int:
        blacks = [(i, j) for i, row in enumerate(picture)
                  for j, x in enumerate(row) if x == "B"]
        row_cnt = Counter(i for i, _ in blacks)
        col_cnt = Counter(j for _, j in blacks)
        return sum(1 for i, j in blacks if row_cnt[i] == 1 and col_cnt[j] == 1)


for pic, expected in examples:
    assert Solution5().findLonelyPixel(pic) == expected
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp:** Time `O(m * n)` để thu thập tọa độ, sau đó `O(B)` với `B` là số điểm đen. Space `O(B)` cho danh sách tọa độ và hai `Counter` (`B` tối đa `m * n`). Nếu đề cho sẵn danh sách tọa độ thay vì ma trận (ví dụ ảnh thưa rất lớn), cách này **không phải đọc cả ma trận**.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (hai mảng đếm)

Bảng trạng thái (ma trận `grid4` ở mục 4, `.` là `W`):

| Cột | Ý nghĩa |
|---|---|
| `(i,j)` | ô `'B'` vừa gặp trong lượt 1 |
| `rows` | mảng đếm số `B` theo hàng sau khi cập nhật |
| `cols` | mảng đếm số `B` theo cột sau khi cập nhật |

Lượt 2 in từng `'B'` cùng hai con số tra được và kết luận.

In [20]:
def trace_count(picture):
    m, n = len(picture), len(picture[0])
    rows, cols = [0] * m, [0] * n
    print("=== LƯỢT 1: đếm ===")
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                rows[i] += 1
                cols[j] += 1
                print(f"gặp B ở ({i},{j}) | rows = {rows} | cols = {cols}")
    print("=== LƯỢT 2: kiểm tra ===")
    ans = 0
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                ok = rows[i] == 1 and cols[j] == 1
                ans += ok
                print(f"({i},{j}): rows[{i}] = {rows[i]}, cols[{j}] = {cols[j]} -> "
                      f"{'CÔ ĐƠN' if ok else 'không'} | ans = {ans}")
    return ans

print("kết quả:", trace_count(grid4))

=== LƯỢT 1: đếm ===
gặp B ở (0,0) | rows = [1, 0, 0, 0] | cols = [1, 0, 0, 0]
gặp B ở (1,2) | rows = [1, 1, 0, 0] | cols = [1, 0, 1, 0]
gặp B ở (2,1) | rows = [1, 1, 1, 0] | cols = [1, 1, 1, 0]
gặp B ở (2,3) | rows = [1, 1, 2, 0] | cols = [1, 1, 1, 1]
=== LƯỢT 2: kiểm tra ===
(0,0): rows[0] = 1, cols[0] = 1 -> CÔ ĐƠN | ans = 1
(1,2): rows[1] = 1, cols[2] = 1 -> CÔ ĐƠN | ans = 2
(2,1): rows[2] = 2, cols[1] = 1 -> không | ans = 2
(2,3): rows[2] = 2, cols[3] = 1 -> không | ans = 2
kết quả: 2


Quan sát: sau lượt 1, `rows = [1, 1, 2, 0]` và `cols = [1, 1, 1, 1]`. Điểm `(2,1)` và `(2,3)` bị loại vì `rows[2] = 2`, hai điểm còn lại có cả hai số bằng 1.

### 8.2. Trace cách 3 (đếm theo cột + `last_row`)

Ta chạy trên ví dụ 1 (ba điểm chéo) **và** một ma trận mà cột có đúng một `B` nhưng hàng của nó lại có nhiều `B`, để thấy bước kiểm tra `rows[last_row[j]]` cần thiết.

In [21]:
def trace_by_column(picture):
    m, n = len(picture), len(picture[0])
    rows, cols, last_row = [0] * m, [0] * n, [0] * n
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                rows[i] += 1
                cols[j] += 1
                last_row[j] = i
    print("sau lượt duyệt: rows =", rows, "| cols =", cols, "| last_row =", last_row)
    ans = 0
    for j in range(n):
        if cols[j] != 1:
            print(f"cột {j}: cols = {cols[j]} (không phải đúng 1) -> bỏ qua")
            continue
        i = last_row[j]
        ok = rows[i] == 1
        ans += ok
        print(f"cột {j}: đúng 1 B ở hàng {i}; rows[{i}] = {rows[i]} -> {'CÔ ĐƠN' if ok else 'không'}")
    return ans

pic_t = [list("BWB"), list("WWW"), list("WBW")]
show(pic_t)
print("kết quả:", trace_by_column(pic_t))

B W B
W W W
W B W
sau lượt duyệt: rows = [2, 0, 1] | cols = [1, 1, 1] | last_row = [0, 2, 0]
cột 0: đúng 1 B ở hàng 0; rows[0] = 2 -> không
cột 1: đúng 1 B ở hàng 2; rows[2] = 1 -> CÔ ĐƠN
cột 2: đúng 1 B ở hàng 0; rows[0] = 2 -> không
kết quả: 1


Trong ví dụ, cột 1 có đúng một `B` ở hàng 2 và hàng 2 chỉ có nó → cô đơn. Cột 0 và cột 2 đều có đúng một `B` nhưng chúng cùng nằm ở hàng 0 (hàng này có 2 `B`) → không cô đơn. Tổng: `1`.

### 8.3. Trace cách 5 (tọa độ + `Counter`)

In [22]:
def trace_sparse(picture):
    blacks = []
    for i, row in enumerate(picture):
        for j, x in enumerate(row):
            if x == "B":
                blacks.append((i, j))
                print(f"thu thập B ở ({i},{j}) -> blacks = {blacks}")
    row_cnt = Counter(i for i, _ in blacks)
    col_cnt = Counter(j for _, j in blacks)
    print("row_cnt =", dict(row_cnt), "| col_cnt =", dict(col_cnt))
    ans = 0
    for i, j in blacks:
        ok = row_cnt[i] == 1 and col_cnt[j] == 1
        ans += ok
        print(f"({i},{j}): hàng có {row_cnt[i]} B, cột có {col_cnt[j]} B -> {'CÔ ĐƠN' if ok else 'không'}")
    return ans

print("kết quả:", trace_sparse(pic_t))

thu thập B ở (0,0) -> blacks = [(0, 0)]
thu thập B ở (0,2) -> blacks = [(0, 0), (0, 2)]
thu thập B ở (2,1) -> blacks = [(0, 0), (0, 2), (2, 1)]
row_cnt = {0: 2, 2: 1} | col_cnt = {0: 1, 2: 1, 1: 1}
(0,0): hàng có 2 B, cột có 1 B -> không
(0,2): hàng có 2 B, cột có 1 B -> không
(2,1): hàng có 1 B, cột có 1 B -> CÔ ĐƠN
kết quả: 1


## 9. Code nộp LeetCode

Chọn **cách 2 (hai mảng đếm)**: dễ hiểu nhất, `O(m * n)` thời gian, `O(m + n)` bộ nhớ. Đây là code đúng class/method/signature (trên LeetCode `List` có sẵn; ở đây ta import ở đầu notebook):

In [23]:
from typing import List


class Solution:
    def findLonelyPixel(self, picture: List[List[str]]) -> int:
        rows = [0] * len(picture)
        cols = [0] * len(picture[0])
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B":
                    rows[i] += 1
                    cols[j] += 1
        ans = 0
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B" and rows[i] == 1 and cols[j] == 1:
                    ans += 1
        return ans


for pic, expected in examples:
    got = Solution().findLonelyPixel(pic)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

3 (kỳ vọng 3 )
0 (kỳ vọng 0 )
1 (kỳ vọng 1 )


## 10. Vì sao thuật toán đúng

**Cách 2.** Gọi `R(i)` là số ký tự `'B'` thực sự trên hàng `i` và `C(j)` là số ký tự `'B'` thực sự trên cột `j`. Lượt 1 duyệt từng ô đúng một lần, mỗi `'B'` ở `(i, j)` cộng 1 vào `rows[i]` và `cols[j]`; do đó sau lượt 1, `rows[i] = R(i)` và `cols[j] = C(j)` (đếm bằng cộng dồn, mỗi `B` được đếm đúng một lần cho hàng của nó và một lần cho cột của nó).

Một ô `'B'` ở `(i, j)` **cô đơn** theo định nghĩa ⇔ không có `'B'` khác ở hàng `i` và không có `'B'` khác ở cột `j`. Vì chính nó đã là một `'B'` trên hàng `i` và trên cột `j`:

- "không có `'B'` khác ở hàng `i`" ⇔ `R(i) = 1`,
- "không có `'B'` khác ở cột `j`" ⇔ `C(j) = 1`.

Hai điều kiện đó chính là điều lượt 2 kiểm tra, nên lượt 2 đếm đúng số điểm cô đơn, mỗi điểm đúng một lần (vì mỗi ô được xét một lần).

**Cách 3.** Điểm cô đơn `(i, j)` nằm trong cột `j` có `C(j) = 1`, và `'B'` duy nhất của cột đó chính là nó. Với mỗi cột có `C(j) = 1`, `last_row[j]` được gán đúng một lần nên bằng hàng của `'B'` duy nhất; kiểm tra `rows[last_row[j]] == 1` đúng bằng điều kiện `R(i) = 1`. Mỗi điểm cô đơn tương ứng với **đúng một** cột nên không đếm trùng; và mỗi cột thỏa điều kiện cho ra đúng một điểm cô đơn.

**Cách 4.** Điểm cô đơn nằm trên hàng có `R(i) = 1`, và khi đó `index("B")` cho đúng cột của nó; điều kiện còn lại `C(j) = 1`. Hàng có `R(i) != 1` không chứa điểm cô đơn (0 `B` thì không có điểm nào; từ 2 `B` trở lên thì mỗi `B` đều có `B` khác cùng hàng).

**Cách 5.** Giống cách 2 nhưng chỉ lưu thống kê cho các hàng/cột có điểm đen (`Counter` trả 0 cho còn lại), và chỉ xét các ô là `'B'` (nằm trong `blacks`).

**Vét cạn (cách 1)** đúng vì là bản dịch nguyên văn của định nghĩa.

**Trường hợp biên.** Không có `'B'` nào: mọi cách trả 0. Ma trận `1 x 1` là `"B"`: `rows = [1]`, `cols = [1]` → 1.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(m * n * (m + n))` | `O(1)` |
| 2. Hai mảng đếm (nộp) | `O(m * n)` | `O(m + n)` |
| 3. Đếm theo cột + `last_row` | `O(m * n)` | `O(m + n)` |
| 4. `count` + `zip` + `index` | `O(m * n)` | `O(m + n)` |
| 5. Tọa độ + `Counter` | `O(m * n)` (đọc ảnh) rồi `O(B)` | `O(B)` |

Với `m = n = 500`: cách 2 chỉ cỡ `2 * 250.000 = 500.000` thao tác; cách 1 có cận trên lý thuyết cỡ hàng trăm triệu (thực tế nhỏ hơn vì có `break`, nhưng với đường chéo ở mục 5 đã thấy số lần đọc tăng theo bình phương). Ta có thể đo thực tế:

In [24]:
def make_grid(m, n, p, rng):
    return [["B" if rng.random() < p else "W" for _ in range(n)] for _ in range(m)]


rng = random.Random(531)
big = make_grid(500, 500, 0.001, rng)     # rất thưa: nhiều điểm cô đơn
dense = make_grid(500, 500, 0.5, rng)     # dày: gần như không có điểm cô đơn

for name, pic in (("thưa", big), ("dày", dense)):
    t0 = time.perf_counter()
    a = Solution().findLonelyPixel(pic)
    t_fast = time.perf_counter() - t0
    t0 = time.perf_counter()
    b = lonely_brute(pic)
    t_slow = time.perf_counter() - t0
    assert a == b
    print(f"ma trận {name}: đáp án {a:>3} | cách 2: {t_fast*1000:7.1f} ms | vét cạn: {t_slow*1000:7.1f} ms")

ma trận thưa: đáp án  86 | cách 2:    15.5 ms | vét cạn:    21.5 ms


ma trận dày: đáp án   0 | cách 2:    26.5 ms | vét cạn:    30.6 ms


Dù vét cạn có thể "thoát sớm" (`break`) nên không đạt đúng cận trên lý thuyết, cách 2 vẫn ổn định ở bậc `O(m * n)` bất kể dữ liệu thế nào.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Ta viết một hàm `check_all` chạy mọi cách giải trên cùng một input và đòi tất cả trùng nhau (cùng đối chiếu với `lonely_brute` làm oracle độc lập).

In [25]:
solvers = [
    ("vét cạn", lonely_brute),
    ("cách 2", Solution2().findLonelyPixel),
    ("cách 3", Solution3().findLonelyPixel),
    ("cách 4", Solution4().findLonelyPixel),
    ("cách 5", Solution5().findLonelyPixel),
]


def check_all(picture, expected=None):
    results = {name: f(picture) for name, f in solvers}
    values = set(results.values())
    assert len(values) == 1, results
    got = values.pop()
    if expected is not None:
        assert got == expected, (got, expected)
    return got


# ví dụ đề
for pic, expected in examples:
    check_all(pic, expected)
print("ví dụ đề: OK")

edge = [
    ([["B"]], 1),                                       # 1 x 1 đen
    ([["W"]], 0),                                       # 1 x 1 trắng
    ([["B", "B"]], 0),                                  # một hàng, hai B
    ([["B", "W", "B"]], 0),                             # cách xa nhưng vẫn cùng hàng
    ([["B"], ["W"], ["B"]], 0),                         # một cột, hai B
    ([["W"] * 4 for _ in range(3)], 0),                 # toàn trắng
    ([["B"] * 3 for _ in range(3)], 0),                 # toàn đen
    ([["B", "W"], ["W", "B"]], 2),                      # hai điểm chéo
    ([["B", "W"], ["B", "W"]], 0),                      # cùng cột
    ([["B", "B"], ["W", "W"]], 0),                      # cùng hàng
    ([["B", "W", "W"], ["W", "W", "B"]], 2),            # ma trận chữ nhật m != n
    ([["W", "B"], ["W", "W"], ["W", "W"]], 1),          # nhiều hàng hơn cột
]
for pic, expected in edge:
    check_all(pic, expected)
print("edge case: OK")

ví dụ đề: OK
edge case: OK


### 12.2. Phản ví dụ cho các suy nghĩ sai

Một vài "heuristic" nghe có lý nhưng sai. Ta chứng minh bằng phản ví dụ cụ thể.

In [26]:
def wrong_neighbors_only(picture):
    """SAI: chỉ xét 4 ô hàng xóm kề bên thay vì cả hàng và cả cột."""
    m, n = len(picture), len(picture[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if picture[i][j] != "B":
                continue
            ok = True
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and picture[a][b] == "B":
                    ok = False
            ans += ok
    return ans


def wrong_rows_only(picture):
    """SAI: chỉ xét hàng, quên cột."""
    return sum(1 for row in picture if row.count("B") == 1)


counter1 = [["B", "W", "B"]]                       # hai B cùng hàng nhưng không kề nhau
counter2 = [["B"], ["W"], ["B"]]                   # hai B cùng cột
print("phản ví dụ 1: đúng =", lonely_brute(counter1), "| 'chỉ hàng xóm' =", wrong_neighbors_only(counter1))
print("phản ví dụ 2: đúng =", lonely_brute(counter2), "| 'chỉ hàng' =", wrong_rows_only(counter2))
assert wrong_neighbors_only(counter1) != lonely_brute(counter1)
assert wrong_rows_only(counter2) != lonely_brute(counter2)

phản ví dụ 1: đúng = 0 | 'chỉ hàng xóm' = 2
phản ví dụ 2: đúng = 0 | 'chỉ hàng' = 2


Phản ví dụ 1: hai `B` cùng hàng nhưng cách nhau một ô, "hàng xóm kề bên" không thấy nhau → sai. Phản ví dụ 2: mỗi hàng đều có đúng một `B` nhưng chúng cùng một cột → chỉ xét hàng sẽ đếm sai.

### 12.3. Đối chiếu nhiều input ngẫu nhiên (seed cố định)

Sinh ma trận ngẫu nhiên **đúng ràng buộc** (`1 <= m, n`; ở đây dùng kích thước nhỏ để vét cạn chạy được) với nhiều mật độ `B` khác nhau, rồi so các cách với nhau và với một oracle viết theo hướng khác (dùng **tập tọa độ**, không dùng bất kỳ mảng đếm nào).

In [27]:
def oracle_by_set(picture):
    """Oracle độc lập: dựa vào tập tọa độ, kiểm tra 'có B nào khác cùng hàng/cột' theo từng điểm."""
    blacks = {(i, j) for i, row in enumerate(picture) for j, x in enumerate(row) if x == "B"}
    ans = 0
    for (i, j) in blacks:
        if not any((a == i or b == j) and (a, b) != (i, j) for (a, b) in blacks):
            ans += 1
    return ans


rng = random.Random(20531)
total = 0
for _ in range(3000):
    m = rng.randint(1, 8)
    n = rng.randint(1, 8)
    p = rng.choice([0.0, 0.05, 0.1, 0.2, 0.35, 0.5, 0.8, 1.0])
    pic = make_grid(m, n, p, rng)
    got = check_all(pic)
    assert got == oracle_by_set(pic)
    total += got
print("3000 ma trận ngẫu nhiên: tất cả cách đồng ý với oracle. Tổng điểm cô đơn gặp:", total)

3000 ma trận ngẫu nhiên: tất cả cách đồng ý với oracle. Tổng điểm cô đơn gặp: 911


In [28]:
# ma trận lớn hơn: đối chiếu 4 cách nhanh với nhau (bỏ vét cạn/oracle chậm)
rng = random.Random(7)
fast = [Solution2().findLonelyPixel, Solution3().findLonelyPixel,
        Solution4().findLonelyPixel, Solution5().findLonelyPixel]
for _ in range(30):
    m = rng.randint(1, 500)
    n = rng.randint(1, 500)
    p = rng.choice([0.0005, 0.002, 0.01, 0.1])
    pic = make_grid(m, n, p, rng)
    vals = {f(pic) for f in fast}
    assert len(vals) == 1
print("30 ma trận lớn (tối đa 500 x 500): các cách nhanh đồng ý")

# trường hợp lớn nhất: toàn trắng, toàn đen, đường chéo 500 x 500
all_w = [["W"] * 500 for _ in range(500)]
all_b = [["B"] * 500 for _ in range(500)]
diag = [["B" if i == j else "W" for j in range(500)] for i in range(500)]
assert Solution().findLonelyPixel(all_w) == 0
assert Solution().findLonelyPixel(all_b) == 0
assert Solution().findLonelyPixel(diag) == 500
print("500 x 500: toàn W ->", 0, "| toàn B ->", 0, "| đường chéo ->", 500)

30 ma trận lớn (tối đa 500 x 500): các cách nhanh đồng ý
500 x 500: toàn W -> 0 | toàn B -> 0 | đường chéo -> 500


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra hàng, quên cột

**Code sai:**

In [29]:
class Wrong1:
    def findLonelyPixel(self, picture):
        rows = [0] * len(picture)
        for i, row in enumerate(picture):
            for x in row:
                if x == "B":
                    rows[i] += 1
        return sum(1 for r in rows if r == 1)          # SAI: không xét cột

pic = [["B", "W"], ["B", "W"]]
print("sai :", Wrong1().findLonelyPixel(pic))
print("đúng:", Solution().findLonelyPixel(pic))

sai : 2
đúng: 0


**Vì sao sai:** định nghĩa đòi hỏi **cả hàng và cột** không có `B` khác. Ở ví dụ trên mỗi hàng chỉ có một `B`, nhưng hai `B` cùng cột nên không điểm nào cô đơn (đáp án đúng là 0). Phải có mảng `cols`.

### Lỗi 2: Dùng `or` thay vì `and`

In [30]:
class Wrong2:
    def findLonelyPixel(self, picture):
        rows = [0] * len(picture)
        cols = [0] * len(picture[0])
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B":
                    rows[i] += 1
                    cols[j] += 1
        ans = 0
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B" and (rows[i] == 1 or cols[j] == 1):   # SAI: phải là and
                    ans += 1
        return ans

pic = [["B", "B"], ["W", "W"]]
print("sai :", Wrong2().findLonelyPixel(pic))
print("đúng:", Solution().findLonelyPixel(pic))

sai : 2
đúng: 0


**Vì sao sai:** `or` chấp nhận điểm chỉ cô đơn theo cột nhưng vẫn có bạn cùng hàng. Ở ví dụ, hai `B` cùng hàng 0 nên đều **không** cô đơn; nhưng mỗi cột lại chỉ có một `B`, thế là `or` đếm sai thành 2. Cần **cả hai** điều kiện: `and`.

### Lỗi 3: Nhầm `m` và `n` khi tạo mảng đếm

In [31]:
class Wrong3:
    def findLonelyPixel(self, picture):
        m, n = len(picture), len(picture[0])
        rows = [0] * n                 # SAI: rows phải có m phần tử
        cols = [0] * m                 # SAI: cols phải có n phần tử
        for i in range(m):
            for j in range(n):
                if picture[i][j] == "B":
                    rows[i] += 1
                    cols[j] += 1
        return sum(1 for i in range(m) for j in range(n)
                   if picture[i][j] == "B" and rows[i] == 1 and cols[j] == 1)

pic = [["W", "W", "B", "W"]]                      # 1 hàng, 4 cột
try:
    Wrong3().findLonelyPixel(pic)
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().findLonelyPixel(pic))

IndexError: list index out of range
đúng: 1


**Vì sao sai:** `rows` được chỉ số bằng `i` (từ `0` đến `m - 1`) nên cần `m` ô; `cols` chỉ số bằng `j` nên cần `n` ô. Với ma trận **không vuông** nhầm lẫn này gây `IndexError` (hoặc kết quả sai thầm lặng khi vô tình không vượt biên). Hãy luôn thử một ma trận `1 x 4` hoặc `4 x 1`.

### Lỗi 4: Quên rằng chính điểm đang xét cũng được đếm

In [32]:
class Wrong4:
    def findLonelyPixel(self, picture):
        rows = [0] * len(picture)
        cols = [0] * len(picture[0])
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B":
                    rows[i] += 1
                    cols[j] += 1
        ans = 0
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B" and rows[i] == 0 and cols[j] == 0:    # SAI: không bao giờ xảy ra
                    ans += 1
        return ans

print("sai :", Wrong4().findLonelyPixel(examples[0][0]))
print("đúng:", Solution().findLonelyPixel(examples[0][0]))

sai : 0
đúng: 3


**Vì sao sai:** `rows[i]` đã tính cả chính điểm `(i, j)`, nên với điểm cô đơn nó bằng **1**, không phải 0. Nếu muốn dùng điều kiện "không có B **khác**" thì phải so với `1`, hoặc trừ đi 1 trước khi so với 0.

### Lỗi 5: So sánh sai ký tự (chữ thường, hoặc so với số)

In [33]:
class Wrong5:
    def findLonelyPixel(self, picture):
        return sum(1 for row in picture for x in row if x == "b")    # SAI: 'b' thường, hoặc 1/0

print("sai :", Wrong5().findLonelyPixel(examples[0][0]))
print("đúng:", Solution().findLonelyPixel(examples[0][0]))

sai : 0
đúng: 3


**Vì sao sai:** ký tự trong đề là chữ **hoa** `'B'`/`'W'` (chuỗi, không phải số `1/0`, không phải chữ thường). Python phân biệt hoa/thường, nên `"b" != "B"` và điều kiện không bao giờ đúng. Hãy sao chép đúng ký tự trong đề.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** đề hỏi một tính chất của phần tử dựa trên "**có phần tử nào khác trong cùng hàng / cùng cột không**" → nghĩ ngay tới **mảng đếm theo hàng và theo cột** (rows/cols counting), tính **một lần**, tra cứu **`O(1)`**.
- **Mẫu chung "precompute rồi lookup":** khi cùng một thống kê bị tính lại nhiều lần cho nhiều đối tượng khác nhau, hãy tính sẵn một lần.
- **Đừng nhìn hàng xóm:** các bài kiểu "kề bên" dùng 4 hoặc 8 hướng; bài này là "cùng hàng/cùng cột" — **toàn bộ** hàng/cột chứ không chỉ ô kề.
- **Kích thước không vuông:** luôn tự hỏi `m` (hàng) và `n` (cột) rồi chọn đúng độ dài cho `rows`, `cols`; thử test `1 x n`, `m x 1`.
- **Điều kiện "khác chính nó":** đếm cả chính nó thì so với `1`.
- **Python:** `zip(*matrix)` lấy cột; `row.count(x)`; `index` cho vị trí đầu tiên; `Counter` trả 0 cho khóa vắng; `sum(1 for ...)` để đếm.
- **Mở rộng tư duy:** hàng/cột có đúng 1 `B` là điều kiện *cần*; ghép hai điều kiện (hàng, cột) bằng `and`.

## 15. Biến thể

1. **Trả về danh sách tọa độ** các điểm cô đơn thay vì chỉ đếm (xem demo bên dưới).
2. **Lonely Pixel II (LeetCode 533, Premium):** thêm số nguyên `target`. Một điểm `'B'` ở `(r, c)` là "cô đơn" khi hàng `r` và cột `c` đều có đúng `target` điểm `'B'`, **và** mọi hàng có `'B'` ở cột `c` đều **giống hệt** hàng `r`. Ý tưởng: đếm theo cột, dùng một `Counter` các hàng (dạng `tuple`/chuỗi) để nhận ra hàng giống nhau. Đối chiếu đề gốc nếu cần.
3. **Điểm cô đơn theo 8 hướng:** `'B'` không có `'B'` nào trong 8 ô kề (bài "chấm đen không kề nhau"). Khi đó mới thật sự là bài xét hàng xóm.
4. **Ảnh rất thưa:** đầu vào là danh sách tọa độ điểm đen (cách 5 áp dụng trực tiếp, `O(B)`).
5. **Cập nhật động:** có thao tác "đổi ô `(i, j)` thành `B`/`W`" xen kẽ truy vấn "đếm điểm cô đơn" → giữ `rows`, `cols` và số điểm cô đơn, cập nhật cục bộ `O(1)` hoặc `O(m + n)`.

Demo biến thể 1 và 3 (đã đối chiếu với vét cạn):

In [34]:
def lonely_positions(picture):
    """Biến thể 1: trả về tọa độ các điểm cô đơn (cách 2)."""
    rows = [r.count("B") for r in picture]
    cols = [c.count("B") for c in zip(*picture)]
    return [(i, j) for i, row in enumerate(picture) for j, x in enumerate(row)
            if x == "B" and rows[i] == 1 and cols[j] == 1]


def isolated_8(picture):
    """Biến thể 3: số 'B' không có 'B' nào trong 8 ô kề."""
    m, n = len(picture), len(picture[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if picture[i][j] != "B":
                continue
            ok = True
            for a in range(max(0, i - 1), min(m, i + 2)):
                for b in range(max(0, j - 1), min(n, j + 2)):
                    if (a, b) != (i, j) and picture[a][b] == "B":
                        ok = False
            ans += ok
    return ans


print(lonely_positions(grid4))
print("8-hướng:", isolated_8(grid4), "| theo hàng/cột:", Solution().findLonelyPixel(grid4))
assert lonely_positions(examples[0][0]) == [(0, 2), (1, 1), (2, 0)]
assert len(lonely_positions(grid4)) == Solution().findLonelyPixel(grid4)

[(0, 0), (1, 2)]
8-hướng: 1 | theo hàng/cột: 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Quét lại hàng và cột cho từng `B` | `O(m * n * (m + n))` | `O(1)` | Chỉ để kiểm chứng trên input nhỏ |
| 2. Hai mảng đếm | Lượt 1 đếm, lượt 2 kiểm tra | `O(m * n)` | `O(m + n)` | **Cách nộp**, rõ ràng, dễ nhớ |
| 3. Đếm theo cột + `last_row` | Chỉ xét cột có đúng 1 `B`, tra hàng của nó | `O(m * n)` | `O(m + n)` | Muốn đọc ma trận đúng một lần |
| 4. `count` + `zip` + `index` | Tận dụng hàm C có sẵn | `O(m * n)` | `O(m + n)` | Viết ngắn, chạy nhanh trong Python |
| 5. Tọa độ + `Counter` | Coi ảnh là tập điểm đen | `O(m * n)` đọc, `O(B)` xử lý | `O(B)` | Ảnh thưa / đầu vào đã là danh sách tọa độ |

Khác nhau thật về ý tưởng: cách 1 không có thống kê; cách 2 có thống kê hàng+cột; cách 3 duyệt theo **cột** thay vì theo ô; cách 4 duyệt theo **hàng** bằng hàm thư viện; cách 5 đổi cách biểu diễn dữ liệu.

In [35]:
# so sánh thời gian nhanh trên ma trận 500 x 500 với mật độ vừa phải
rng = random.Random(16)
pic = make_grid(500, 500, 0.002, rng)
ref = Solution2().findLonelyPixel(pic)
for name, f in solvers[1:]:
    t0 = time.perf_counter()
    got = f(pic)
    ms = (time.perf_counter() - t0) * 1000
    assert got == ref
    print(f"{name}: đáp án {got}, {ms:6.1f} ms")

cách 2: đáp án 65,   15.5 ms
cách 3: đáp án 65,   10.5 ms


cách 4: đáp án 65,   12.1 ms
cách 5: đáp án 65,   10.9 ms


## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [36]:
print(Solution().findLonelyPixel([["B", "W"], ["W", "B"]]))
print(Solution().findLonelyPixel([["B", "B"], ["W", "B"]]))
print(Solution().findLonelyPixel([["W", "W"], ["W", "W"]]))

2
0
0


**Bài 2 (tính tay).** Với ma trận sau (`.` là `W`):

```
B . . B
. . . .
. B . .
. . B .
```

hãy tìm các mảng `rows` (4 số) và `cols` (4 số), rồi suy ra đáp án. Gán vào `ans2_rows`, `ans2_cols`, `ans2` (đang là `None`).

In [37]:
ans2_rows = None   # TODO: list 4 số
ans2_cols = None   # TODO: list 4 số
ans2 = None        # TODO: số điểm cô đơn

**Bài 3 (điền TODO).** Hoàn thành hàm đếm số **hàng có đúng một `B`** *và* số **cột có đúng một `B`**, trả về cặp `(số_hàng, số_cột)`.

In [38]:
def single_rows_cols(picture):
    # TODO: dùng row.count("B") và zip(*picture)
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_lonely(picture)` theo cách 3 (đếm theo cột + `last_row`).

In [39]:
def my_lonely(picture):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `lonely_with_target(picture, target)` đếm các `'B'` mà hàng của nó và cột của nó đều có **đúng `target`** điểm `'B'` (bản nới lỏng, **không** có điều kiện "các hàng giống nhau" của bài 533). Với `target = 1` phải trùng đáp án bài này. (b) Viết `max_lonely_after_one_flip(picture)`: được đổi **đúng một** ô bất kỳ (từ `W` thành `B` hoặc từ `B` thành `W`), số điểm cô đơn lớn nhất có thể đạt được là bao nhiêu? (Chỉ cần cách vét cạn: thử mọi ô, mỗi lần gọi lại `Solution`. Bắt buộc đổi đúng một ô, không được "không đổi".)

In [40]:
def lonely_with_target(picture, target):
    # TODO
    pass

def max_lonely_after_one_flip(picture):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: vẽ ra từng ma trận và gạch các `B` có bạn cùng hàng/cột.
- Bài 2: đếm số `B` của từng hàng, rồi của từng cột; hai điểm `B` hàng 0 cùng nằm trên một hàng.
- Bài 3: `row.count("B") == 1` cho hàng; cột lấy bằng `zip(*picture)`.
- Bài 4: nhớ `last_row[j] = i` mỗi khi gặp `B`.
- Bài 5: thay số `1` bằng biến `target`; với (b) chỉ cần hai vòng `for` đổi ô rồi **đổi lại**.

**Mức 2 (vừa):**
- Bài 2: `rows = [2, 0, 1, 1]`; mỗi cột chỉ có đúng một `B` nên `cols` toàn số 1. Hàng nào có đúng một `B` thì mới có thể cô đơn.
- Bài 3: `sum(1 for row in picture if row.count("B") == 1)` và `sum(1 for col in zip(*picture) if col.count("B") == 1)`.
- Bài 4: sau lượt duyệt, `sum(1 for j in range(n) if cols[j] == 1 and rows[last_row[j]] == 1)`.
- Bài 5(b): dùng bản sao ma trận (`[row[:] for row in picture]`) hoặc đổi rồi đổi lại để không phá input.

**Mức 3 (gần đáp án):**
- Bài 3: xem cách 4 ở mục 7.3.
- Bài 4: xem mục 7.2, code nộp cách 3.
- Bài 5(a): `rows[i] == target and cols[j] == target`; (b): `best = max(best, Solution().findLonelyPixel(picture))` sau mỗi lần đổi ô, rồi đổi lại ô về giá trị cũ.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [41]:
# Bài 1
assert Solution().findLonelyPixel([["B", "W"], ["W", "B"]]) == 2
assert Solution().findLonelyPixel([["B", "B"], ["W", "B"]]) == 0
assert Solution().findLonelyPixel([["W", "W"], ["W", "W"]]) == 0

# Bài 2
ex2 = [list("B..B"), list("...."), list(".B.."), list("..B.")]
ex2 = [["W" if x == "." else x for x in row] for row in ex2]
ans2_rows = [2, 0, 1, 1]
ans2_cols = [1, 1, 1, 1]
ans2 = 2
assert ans2_rows == [r.count("B") for r in ex2]
assert ans2_cols == [c.count("B") for c in zip(*ex2)]
assert ans2 == Solution().findLonelyPixel(ex2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [42]:
# Bài 3
def single_rows_cols(picture):
    sr = sum(1 for row in picture if row.count("B") == 1)
    sc = sum(1 for col in zip(*picture) if col.count("B") == 1)
    return (sr, sc)

assert single_rows_cols(ex2) == (2, 4)
assert single_rows_cols(examples[0][0]) == (3, 3)
assert single_rows_cols([["W", "W"]]) == (0, 0)
assert single_rows_cols([["B", "B"]]) == (0, 2)
print("Bài 3 OK")

Bài 3 OK


In [43]:
# Bài 4
def my_lonely(picture):
    m, n = len(picture), len(picture[0])
    rows, cols, last_row = [0] * m, [0] * n, [0] * n
    for i in range(m):
        for j in range(n):
            if picture[i][j] == "B":
                rows[i] += 1
                cols[j] += 1
                last_row[j] = i
    return sum(1 for j in range(n) if cols[j] == 1 and rows[last_row[j]] == 1)

rng = random.Random(4)
for _ in range(500):
    pic = make_grid(rng.randint(1, 7), rng.randint(1, 7), rng.choice([0.1, 0.3, 0.6]), rng)
    assert my_lonely(pic) == lonely_brute(pic)
print("Bài 4 OK")

Bài 4 OK


In [44]:
# Bài 5
def lonely_with_target(picture, target):
    rows = [r.count("B") for r in picture]
    cols = [c.count("B") for c in zip(*picture)]
    return sum(1 for i, row in enumerate(picture) for j, x in enumerate(row)
               if x == "B" and rows[i] == target and cols[j] == target)


def max_lonely_after_one_flip(picture):
    work = [row[:] for row in picture]          # bản sao để không phá input
    best = 0
    for i in range(len(work)):
        for j in range(len(work[0])):
            old = work[i][j]
            work[i][j] = "W" if old == "B" else "B"
            best = max(best, Solution().findLonelyPixel(work))
            work[i][j] = old
    return best


# (a) target = 1 trùng bài gốc; target = 2 với ví dụ nhỏ
assert lonely_with_target(examples[0][0], 1) == 3
assert lonely_with_target(examples[1][0], 1) == 0
assert lonely_with_target([["B", "B"], ["B", "B"]], 2) == 4
assert lonely_with_target([["B", "B"], ["W", "W"]], 2) == 0

# (b) 1 hàng 'B W B': đổi một B thành W -> còn 1 B cô đơn
assert max_lonely_after_one_flip([["B", "W", "B"]]) == 1
# toàn W: đổi một ô thành B -> 1 điểm cô đơn
assert max_lonely_after_one_flip([["W", "W"], ["W", "W"]]) == 1
# đã có 2 điểm cô đơn chéo nhưng BẮT BUỘC đổi đúng một ô -> tối đa còn 1
assert max_lonely_after_one_flip([["B", "W"], ["W", "B"]]) == 1
# phần (a) so với brute trên ngẫu nhiên
rng = random.Random(5)
for _ in range(300):
    pic = make_grid(rng.randint(1, 6), rng.randint(1, 6), rng.choice([0.1, 0.3, 0.6]), rng)
    assert lonely_with_target(pic, 1) == lonely_brute(pic)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đếm hàng/cột rồi kiểm tra":**

```python
class Solution:
    def findLonelyPixel(self, picture: List[List[str]]) -> int:
        rows = [0] * len(picture)          # m phần tử
        cols = [0] * len(picture[0])       # n phần tử
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B":
                    rows[i] += 1
                    cols[j] += 1
        ans = 0
        for i, row in enumerate(picture):
            for j, x in enumerate(row):
                if x == "B" and rows[i] == 1 and cols[j] == 1:
                    ans += 1
        return ans
```

**Template ngắn (Pythonic):**

```python
cols = [c.count("B") for c in zip(*picture)]
return sum(1 for row in picture if row.count("B") == 1 and cols[row.index("B")] == 1)
```

**Checklist trước khi submit:**

- [ ] `rows` có `m` phần tử, `cols` có `n` phần tử (thử ma trận `1 x n` và `m x 1`)?
- [ ] Điều kiện là `rows[i] == 1 and cols[j] == 1` (dùng `and`, so với `1` vì đã đếm chính nó)?
- [ ] Chỉ đếm khi ô là `'B'` (chữ hoa, kiểu chuỗi)?
- [ ] Không có `'B'` nào -> trả 0; toàn `'B'` -> 0 (trừ `1 x 1`)?
- [ ] Không sửa input (nếu chỉ đọc thì không cần lo)?
- [ ] Độ phức tạp `O(m * n)` đủ cho `500 x 500`?
- [ ] Đã so với vét cạn trên input nhỏ ngẫu nhiên?